# Projet : Prédiction du Diabète  
## Objectifs  
Ce projet vise à :  
- Construire un modèle de Machine Learning capable de prédire si un patient est diabétique ou non.  
- Explorer les facteurs influençant le diabète à partir des données fournies.


In [ ]:
!pip install pandas scikit-learn comet_ml numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 710.6/710.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 980.3/980.3 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: python-box
    Found existing installation: python-box 7.3.0
    Uninstalling python-box-7.3.0:
      Successfully uninstalled python-box-7.3.0


In [ ]:
import pandas as pd
import kagglehub

#df = pd.read_csv("diabetes_ds.csv")
path = kagglehub.dataset_download("iammustafatz/diabetes-prediction-dataset")
df = pd.read_csv(path + "/diabetes_prediction_dataset.csv")

100%|██████████| 734k/734k [00:00<00:00, 20.3MB/s]

Extracting files...


## 1. Préparation des données pour l'entrainement

In [ ]:
# On retire la variable smoking_history de la table, car, d’après les études qu’on a effectuées sur les graphes,
# elle n’a pas d’impact significatif.
df1 = df.copy(deep=True).drop(['smoking_history'], axis=1)

In [ ]:
# Remplacement des valeurs dans la colonne du genre
df1.replace({'Male': 0, 'Female': 1, 'Other': 2}, inplace=True)

<ipython-input-5-af05e1c85d37>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df1.replace({'Male': 0, 'Female': 1, 'Other': 2}, inplace=True)


In [ ]:
# Filtrage entre les diabétiques et les non-diabétiques
diabetic_patients = df1[df1['diabetes'] == 1]
non_diabetic_patients = df1[df1['diabetes'] == 0]

In [ ]:
# Sample la quantité de non_diabetic_patients avec la lenght max des diabetic_patients
non_diabetic_patients = non_diabetic_patients.sample(n=len(diabetic_patients), random_state=42)

In [ ]:
# Création d'un model avec toutes les catégories dedans
balanced_df = pd.concat([diabetic_patients, non_diabetic_patients])

# 2. Utilisation des pipelines pour le préprocessing


In [ ]:
from sklearn.compose import make_column_selector
import numpy as np

# Liste des colonnes à conserver dans le DataFrame
keepCols = ["gender", "hypertension", "heart_disease", "age", "bmi", "HbA1c_level", "blood_glucose_level", "diabetes"]

# Filtrage du DataFrame pour ne conserver que les colonnes spécifiées
balanced_df = balanced_df[keepCols]

# Sélecteur pour les colonnes catégorielles (dtype 'object')
cat_selector = make_column_selector(dtype_include=object)

# Sélecteur pour les colonnes numériques, en s'assurant que le nom de la colonne figure dans 'keepCols'
num_selector = make_column_selector(dtype_include=np.number, pattern=f"({'|'.join(keepCols)})")

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OrdinalEncoder

# Pour les "objects", je les encode avec l'ordinal encoder (= chaque catégorie va devenir un nombre)
cat_tree_processor = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-2,
)

# Pour les "numeric" je choisis de remplacer les valeurs absentes par la moyenne
num_tree_processor = SimpleImputer(strategy="mean", add_indicator=True)

### **Scaler**

L'utilisation d'un scaler permet d'améliorer les performances, la vitesse et l'interprétabilité du model.

Il est particulièrement utile pour équilibrer le processus d'apprentissage de notre model en mettent les caractéristiques sur une même échelle ce qui peut aider le model à apprendre des relations plus significatives entre les caractéristiques et la variable cible (diabete)

**Exemple :**

Prenons l'âge (entre 20 et 80 ans) et le niveau de glucose dans le sang (entre 50 et 500 mg/dL). Le niveau de glucose pourrait dominer l'apprentissage du model en raison de sa plage de valeurs plus large.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler

num_tree_processor = make_pipeline(
    StandardScaler(),  # Ajout de StandardScaler
    SimpleImputer(strategy="mean", add_indicator=True)
)

# J'assemble le tout pour faire une première pipeline de preprocessing et j'affiche le résumé
tree_preprocessor = make_column_transformer(
    (num_tree_processor, num_selector), (cat_tree_processor, cat_selector)
)

tree_preprocessor

ColumnTransformer(transformers=[('pipeline',
                                 Pipeline(steps=[('standardscaler',
                                                  StandardScaler()),
                                                 ('simpleimputer',
                                                  SimpleImputer(add_indicator=True))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x79fc052dc580>),
                                ('ordinalencoder',
                                 OrdinalEncoder(encoded_missing_value=-2,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x79fbc985fca0>)])

In [ ]:
from sklearn.model_selection import train_test_split

# Enregistrement des colonnes et de la cible
features = ["gender", "hypertension", "heart_disease", "age", "bmi", "HbA1c_level", "blood_glucose_level"]
target = "diabetes"

X = balanced_df[features]
y = balanced_df[target]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Création du modèle avec un maximum de 500 itérations pour l'optimisation
clf = LogisticRegression(max_iter=500)

# Entraînement du modèle sur les données
clf.fit(x_train, y_train)

LogisticRegression(max_iter=500)

In [ ]:
# print("coef:", clf.coef_)


# print(balanced_df.diabetes.value_counts())

# print("Train score :", clf.score(x_train, y_train) * 100)
# print("Test  score :", clf.score(x_test, y_test) * 100)

# print("-"*10)

# print(x_test[0:1])
# print("-"*10)

On utilise une pipeline avec **StandardScaler**, **PCA**, et **SVC** pour tester plusieurs hyperparamètres (nombre de composantes **PCA**, **C**, et **kernel** du **SVC**) à l'aide de **GridSearchCV**, afin d'optimiser la précision sur mes données d'entraînement.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
import numpy as np

# Création de la pipeline
pipe = Pipeline([
        ('scale', StandardScaler()),
        ('reduce_dims', PCA(n_components=4)),
        ('clf', SVC(kernel='linear', C=1))
])

# Grille de recherche pour optimiser les hyperparamètres
param_grid = dict(
    reduce_dims__n_components=[4, 6, 8],
    clf__C=np.logspace(-4, 1, 6),
    clf__kernel=['rbf', 'linear']
)

# GridSearchCV pour effectuer la recherche hyperparamétrique avec validation croisée
grid = GridSearchCV(
    pipe,  # pipeline à optimiser
    param_grid=param_grid,  # grille d'hyperparamètres
    cv=3,  # Validation croisée avec 3 plis
    n_jobs=1,  # Nombre de cœurs du processeur à utiliser pour le calcul
    verbose=2,  # Niveau de verbosité pour afficher les étapes du processus
    scoring='accuracy'  # indique que la précision sera utilisée comme métrique de performance pour choisir les meilleurs paramètres
)

# Entraînement du modèle avec GridSearchCV sur les données d'entraînement
grid.fit(x_train, y_train)

# Affichage des résultats trouvés
print(grid.best_score_)
print(grid.cv_results_)


Fitting 3 folds for each of 36 candidates, totalling 108 fits
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=4; total time=   5.9s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=4; total time=   7.5s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=4; total time=   6.1s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=6; total time=   8.0s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=6; total time=   6.3s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=6; total time=   8.4s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=8; total time=   0.0s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=8; total time=   0.0s
[CV] END clf__C=0.0001, clf__kernel=rbf, reduce_dims__n_components=8; total time=   0.0s
[CV] END clf__C=0.0001, clf__kernel=linear, reduce_dims__n_components=4; total time=   3.1s
[CV] END clf__C=0.0001, clf__kernel=linear, r

/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
36 fits failed out of a total of 108.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/sklearn/pipeline.py", line 652, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "/usr/local/lib/python3.

0.8953680135475334
{'mean_fit_time': array([4.14767456, 4.74402563, 0.0095071 , 2.52886391, 2.91840768,
       0.0095245 , 4.27221696, 4.60870266, 0.00966835, 1.75381772,
       1.42981164, 0.00804456, 1.96004732, 1.95150566, 0.00878541,
       1.26116538, 0.81131514, 0.00866318, 1.01554092, 1.52501742,
       0.01309752, 0.82167737, 0.86388326, 0.00896827, 1.24285587,
       1.09256275, 0.00731715, 1.10691325, 1.58157468, 0.00930985,
       1.350993  , 1.79632195, 0.00899744, 2.14806072, 2.93374014,
       0.0118049 ]), 'std_fit_time': array([1.64237127e-01, 4.06914228e-01, 7.36695260e-04, 1.26062089e-01,
       4.55689522e-01, 4.07234629e-05, 4.23406693e-01, 2.92614706e-01,
       5.72066721e-04, 3.87497390e-01, 1.13405931e-02, 8.84338623e-04,
       4.96722409e-01, 5.08519389e-02, 5.03933650e-04, 2.50413472e-01,
       1.45931109e-02, 5.95716799e-04, 2.24805009e-02, 3.40621487e-01,
       2.89576305e-03, 5.30979487e-02, 7.52364683e-02, 2.79258990e-04,
       3.28648010e-01, 3.509838

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

# Création d'un pipeline avec un préprocesseur (tree_preprocessor) et un modèle de régression logistique
rfr_pipeline = make_pipeline(
    tree_preprocessor,  # Étape de prétraitement des données
    clf
)

rfr_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline',
                                                  Pipeline(steps=[('standardscaler',
                                                                   StandardScaler()),
                                                                  ('simpleimputer',
                                                                   SimpleImputer(add_indicator=True))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79fc052dc580>),
                                                 ('ordinalencoder',
                                                  OrdinalEncoder(encoded_missing_value=-2,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79fbc985fca0>)])),
                ('logisticregression', LogisticRegression(max_iter=500))])

#Configuration du projet Comet ML

On configure le projet Comet ML pour suivre les expériences d'entraînement du modèle



In [ ]:
import os

import comet_ml
from comet_ml.integration.sklearn import load_model, log_model

import cloudpickle
from sklearn import datasets, ensemble
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Configuration de Comet ML
WORKSPACE = "justrunnz"
MODEL_NAME = "diabetes-predict-model"
PROJECT_NAME = "diabetes-predict"

# Connexion à Comet ML
comet_ml.login()

# Début d'une nouvelle expérimentation Comet
experiment = comet_ml.start(
    api_key="6l3PPIsKeGgBrUF4d5Lv0XKmW",
    project_name=PROJECT_NAME,
    workspace=WORKSPACE,
)

# Configuration de l'expérimentation
experiment.set_name("DiabetesPredict_2")
experiment.add_tag("DiabetesPredict_2")

# Entraînement du pipeline
rfr_pipeline.fit(x_train, y_train)

# Enregistrement de la pipeline dans Comet
log_model(experiment, MODEL_NAME, rfr_pipeline, persistence_module=cloudpickle)
experiment.register_model(MODEL_NAME)

# Fin de l'expérimentation
experiment.end()

COMET WARNING: Ending the running experiment and creating a new Experiment because:
workspace doesn't match ('justrunnz' != 'None')
project_name doesn't match ('diabetes-predict' != 'None')
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn.
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml ExistingExperiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : DiabetesPredict
COMET INFO:     url                   : https://www.comet.com/justrunnz/diabetes-predict/ae0f79f9b30343de850bc40de6f9acbf
COMET INFO: 
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment

##Prediction avec DataFrame

In [ ]:
import comet_ml
from comet_ml import API
from comet_ml.integration.sklearn import load_model, log_model
import pandas as pd

# Configuration de Comet ML
WORKSPACE = "justrunnz"
MODEL_NAME = "diabetes-predict-model"
MODEL_VERSION = "1.2.0"

# Connexion à Comet ML
comet_ml.login()

# Reprise d'une expérimentation existante via la key unique
experiment = comet_ml.start(experiment_key="2d98664f833d477e93de91acf49bf5e8")

# Chargement du modèle depuis le registre de modèles Comet
loaded_model = load_model(f"registry://{WORKSPACE}/{MODEL_NAME}")

# Préparation des nouvelles données à prédire
new_data = pd.DataFrame({
    'gender': [0],  # Sexe (exemple: 0 = femme, 1 = homme)
    'hypertension': [0],  # Hypertension (0 = non, 1 = oui)
    'heart_disease': [0],  # Maladie cardiaque (0 = non, 1 = oui)
    'age': [50.0],
    'bmi': [25.0],
    'HbA1c_level': [600.0],
    'blood_glucose_level': [100]
})

# Utilisation du modèle pour faire une prédiction
prediction = loaded_model.predict(new_data)

print(f"Prediction for new data: {prediction}")


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/justrunnz/diabetes-predict/2d98664f833d477e93de91acf49bf5e8

COMET INFO: Remote Model 'justrunnz/diabetes-predict-model:None' download has been started asynchronously.
COMET INFO: Still downloading 2 file(s), remaining 3.30 KB/3.30 KB
COMET INFO: Remote Model 'justrunnz/diabetes-predict-model:None' has been successfully downloaded.
COMET INFO: Downloaded asset files is in '/tmp/tmp1xt_1d5m' folder.


Prediction for new data: [1]
